# Pipeline for the Naive Model using AIII - A

## Imports

In [1]:
import sys
import os
import numpy as np
from scipy.linalg import expm

# Add TFIM directory to path so build_TFIM is importable
sys.path.insert(0, os.path.join(os.getcwd(), "..", "TFIM"))

from decompose import decompose
from single_qubit_decomposer import to_gates
from build_TFIM import TFIM_Ham


## Construct the Unitary

In [2]:
# Model parameters
n = 4
J = 1.0
h = 1.0
t = 0.5
rotated = True
periodic = False

print(f"n = {n}, J = {J}, h = {h}, t = {t}, rotated = {rotated}, periodic = {periodic}")

H = TFIM_Ham(n, J=J, h=h, rotated=rotated, periodic=periodic)
U_t = expm(-1j * H * t)
print(f"H shape: {H.shape}")
print(f"U(t) shape: {U_t.shape}, unitary check: {np.allclose(U_t @ U_t.conj().T, np.eye(2**n))}")


n = 4, J = 1.0, h = 1.0, t = 0.5, rotated = True, periodic = False
H shape: (16, 16)
U(t) shape: (16, 16), unitary check: True


## Decompose the Unitary

In [3]:
ops = decompose(U_t)
print(f"Number of ops: {len(ops)}")

from collections import Counter
counts = Counter(op_type for _, _, op_type in ops)
print(f"By type: {dict(counts)}")


Number of ops: 127
By type: {'sq': 64, 'rz': 42, 'ry': 21}


## Convert to gates

In [4]:
gates = to_gates(ops, n)
print(f"Number of gates: {len(gates)}")
print(gates[:8])

print(f"Number of gates by type: {Counter(name for name, _, _ in gates)}")

Number of gates: 550
[('Rz', -2.1392078399108807, [3]), ('Ry', 2.1006026493877856, [3]), ('Rz', 2.98732654889445, [3]), ('Rz', 0.3521909824927961, [2]), ('CNOT', None, [3, 2]), ('Rz', 0.6659126110636638, [2]), ('CNOT', None, [3, 2]), ('Rz', -4.8034714483773016, [3])]
Number of gates by type: Counter({'Rz': 234, 'CNOT': 204, 'Ry': 112})


## Verify Circuit

In [5]:
import pennylane as qml

dev = qml.device("default.qubit", wires=n)

@qml.qnode(dev)
def circuit():
    for name, angle, wires in gates:
        if name == "Ry":
            qml.RY(angle, wires=wires[0])

        elif name == "Rz":   
            qml.RZ(angle, wires=wires[0])

        elif name == "CNOT": 
            qml.CNOT(wires=wires)
            
    return qml.state()

U_circuit = qml.matrix(circuit)()

phase = np.angle(np.trace(U_circuit @ U_t.conj().T))
err = np.max(np.abs(U_circuit - np.exp(1j * phase) * U_t))
print(f"Max error (phase-aligned): {err:.2e}")
print("PASS" if err < 1e-6 else "FAIL")

Max error (phase-aligned): 2.60e-15
PASS
